# День 1 — Архитектура трансформеров и токенизация

**Цель дня:** понять, как работают трансформеры, и научиться превращать текст в токены.

Этот ноутбук содержит примеры из всех задач Дня 1. Переиспользуемые функции вынесены в модуль [`tokenization_utils.py`](tokenization_utils.py) — они понадобятся в День 2 для работы с моделью.

## Задача 1: Установка библиотек

Если библиотеки ещё не установлены, раскомментируйте и выполните ячейку ниже (или `pip install -r requirements.txt`).

In [1]:
# !pip install transformers torch

In [2]:
from transformers import AutoTokenizer
import torch

print(f"torch: {torch.__version__}")

C:\Users\Vsevolod\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.13.0+cpu


## Задача 2: Загрузка токенизатора

In [5]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"vocab_size: {tokenizer.vocab_size}")
print(f"model_max_length: {tokenizer.model_max_length}")

vocab_size: 30522
model_max_length: 512


## Задача 3: Токенизация текста

In [7]:
text = "This movie was absolutely amazing!"

tokens = tokenizer(text)
print(tokens)

{'input_ids': [101, 2023, 3185, 2001, 7078, 6429, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


In [8]:
input_ids = tokens['input_ids']
print(f'Количество токенов: {len(input_ids)}')

decoded = tokenizer.decode(input_ids)
print(f'Декодировано: {decoded}')

Количество токенов: 8
Декодировано: [CLS] this movie was absolutely amazing! [SEP]


Обратите внимание: при декодировании появились специальные токены `[CLS]` и `[SEP]`, которые токенизатор добавил автоматически в начало и конец.

## Задача 4: Работа с батчами

Функция `tokenize_texts` находится в модуле `tokenization_utils.py`. Здесь показываем и локальное определение (для наглядности), и импорт из модуля.

In [9]:
def tokenize_texts(texts, max_length=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )


texts = [
    "This movie was great!",
    "Terrible movie, waste of time.",
]

batch = tokenize_texts(texts)
print(f'Shape: {batch["input_ids"].shape}')

Shape: torch.Size([2, 9])


In [10]:
print(f'Attention mask:\n{batch["attention_mask"]}')

Attention mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])


`attention_mask` содержит 1 там, где реальные токены, и 0 на паддинге. Более короткий текст дополнен нулями до длины более длинного.

In [11]:
# То же самое, но через переиспользуемый модуль
from tokenization_utils import tokenize_texts as tokenize_texts_util

batch2 = tokenize_texts_util(texts, tokenizer)
print(f'Shape: {batch2["input_ids"].shape}')

Shape: torch.Size([2, 9])


## Задача 5: Специальные токены

In [12]:
print(f'CLS token: {tokenizer.cls_token} (ID: {tokenizer.cls_token_id})')
print(f'SEP token: {tokenizer.sep_token} (ID: {tokenizer.sep_token_id})')
print(f'PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})')

CLS token: [CLS] (ID: 101)
SEP token: [SEP] (ID: 102)
PAD token: [PAD] (ID: 0)


In [13]:
single = tokenizer(text, return_tensors="pt")
print(f'Input IDs: {single["input_ids"]}')
print(f'Decoded: {tokenizer.decode(single["input_ids"][0])}')

Input IDs: tensor([[ 101, 2023, 3185, 2001, 7078, 6429,  999,  102]])
Decoded: [CLS] this movie was absolutely amazing! [SEP]


## Задача 6: Функция explain_tokenization

Тоже вынесена в `tokenization_utils.py`. Показывает, как текст разбивается на токени (subword-токенизация).

In [14]:
def explain_tokenization(text, tokenizer):
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    print(f'Исходный текст: {text}')
    print(f'Токены: {tokens}')
    print(f'IDs: {ids}')
    print(f'Количество: {len(tokens)}')


explain_tokenization("Transformers are amazing!", tokenizer)

Исходный текст: Transformers are amazing!
Токены: ['transformers', 'are', 'amazing', '!']
IDs: [19081, 2024, 6429, 999]
Количество: 4


Слово `Transformers` разбилось на subword-токены (например `transform` + `##ers`) — так модель справляется со словами, которых нет целиком в словаре.

Попробуем ещё несколько примеров и версию из модуля:

In [12]:
from tokenization_utils import explain_tokenization as explain_util

for t in ["tokenization", "unbelievable", "GPT models are powerful"]:
    explain_util(t, tokenizer)
    print("-" * 40)

Исходный текст: tokenization
Токены: ['token', '##ization']
IDs: [19204, 3989]
Количество: 2
----------------------------------------
Исходный текст: unbelievable
Токены: ['unbelievable']
IDs: [23653]
Количество: 1
----------------------------------------
Исходный текст: GPT models are powerful
Токены: ['gp', '##t', 'models', 'are', 'powerful']
IDs: [14246, 2102, 4275, 2024, 3928]
Количество: 5
----------------------------------------


## Чекпоинт

К концу дня у нас есть:
- ✅ Загруженный токенизатор
- ✅ Понимание, как текст превращается в токены
- ✅ Функция `tokenize_texts` для батчей
- ✅ Функция `explain_tokenization`

Переиспользуемый код — в [`tokenization_utils.py`](tokenization_utils.py). В День 2 подключим модель.